# Project 1: Automatic Review Analyzer

This notebook is the experimental and visualization layer for Unit 1 Project 1.

We compare **Perceptron, Average Perceptron, and Pegasos**. The reusable learning algorithms live in `linear_classification.py`; review-specific text processing stays in this notebook.

The complete project story is:

**reviews → vocabulary → sparse feature vectors X + labels y → three linear classifiers → validation → best `lambda` → word weights → classifier-specific decision-boundary visualizations.**


In [ ]:
from pathlib import Path
import re
import sys
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != 'project_1':
    candidate = PROJECT_DIR / 'unit_1' / 'project_1'
    if candidate.exists():
        PROJECT_DIR = candidate
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from linear_classification import accuracy, average_perceptron, pegasos, perceptron


## 1. Convert review text into vectors

For a vocabulary

$$
V=\{w_1,w_2,\ldots,w_d\},
$$

each review becomes a sparse feature vector

$$
x_i\in\mathbb{R}^d,
$$

and the labels form a separate vector

$$
y\in\{-1,+1\}^m.
$$

Keeping `X` and `y` separate makes the machine-learning representation explicit: `X` contains features and `y` contains targets. The reusable classifiers still receive `(y_i, x_i)` pairs because that is the module's sparse-data interface.


In [ ]:
TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

def build_vocabulary(reviews):
    words = {token for _, text in reviews for token in tokenize(text)}
    return {word: i for i, word in enumerate(sorted(words))}

def extract_features(text, vocabulary):
    features = {}
    for token in tokenize(text):
        index = vocabulary.get(token)
        if index is not None:
            features[index] = 1.0
    return features

def vectorize(reviews, vocabulary):
    X = [extract_features(text, vocabulary) for _, text in reviews]
    y = np.array([label for label, _ in reviews], dtype=int)
    return X, y

reviews = [(+1, 'great product'), (+1, 'excellent product'), (+1, 'good value'), (+1, 'great quality'), (+1, 'excellent value'), (-1, 'bad product'), (-1, 'terrible product'), (-1, 'poor value'), (-1, 'bad quality'), (-1, 'terrible value')]
train_reviews = reviews[:8]
validation_reviews = reviews[8:]
vocabulary = build_vocabulary(train_reviews)
X_train, y_train = vectorize(train_reviews, vocabulary)
X_validation, y_validation = vectorize(validation_reviews, vocabulary)
train_data = list(zip(y_train, X_train))
validation_data = list(zip(y_validation, X_validation))

print('Vocabulary:', vocabulary)
print('X_train[0]:', X_train[0])
print('y_train:', y_train)
print('First training pair:', train_data[0])


## 2. Train the three classifiers

For a review vector `x`, the linear score is

$$
f(x;\theta)=\theta^T x
$$

and

$$
\hat y=\mathrm{sign}(\theta^T x).
$$

The three algorithms use the same representation but different learning rules.


In [ ]:
models = {'Perceptron': perceptron(train_data, epochs=20), 'Average Perceptron': average_perceptron(train_data, epochs=20), 'Pegasos': pegasos(train_data, lambda_=1e-3, epochs=40, seed=3)}
for name, weights in models.items():
    print(f'{name:20} train={accuracy(weights, train_data):.3f} validation={accuracy(weights, validation_data):.3f}')


## 3. Select the best Pegasos regularization parameter

Pegasos uses

$$
J(\theta)=\frac{\lambda}{2}\|\theta\|^2+\frac{1}{m}\sum_{i=1}^{m}\max\{0,1-y_i\theta^Tx_i\}.
$$

Its learning rate is

$$
\eta_t=\frac{1}{\lambda t}.
$$

We select the `lambda` that gives maximum validation accuracy:

$$
\lambda^*=\arg\max_{\lambda}\mathrm{ValidationAccuracy}(\lambda).
$$


In [ ]:
lambda_grid = np.array([1e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1])
validation_scores = []
for lambda_ in lambda_grid:
    weights = pegasos(train_data, lambda_=lambda_, epochs=40, seed=3)
    score = accuracy(weights, validation_data)
    validation_scores.append(score)
    print(f'lambda={lambda_:>8.1e} | validation accuracy={score:.3f}')
best_score = max(validation_scores)
best_lambda = float(lambda_grid[np.flatnonzero(np.isclose(validation_scores, best_score))[0]])
print(f'\nSelected lambda*: {best_lambda:.1e}')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(lambda_grid, validation_scores, marker='o', linewidth=1.5)
ax.scatter([best_lambda], [best_score], s=100, zorder=3, label=f'lambda* = {best_lambda:.1e}')
ax.axvline(best_lambda, linestyle='--', linewidth=1)
for lambda_, score in zip(lambda_grid, validation_scores):
    ax.annotate(f'{lambda_:.1e}', (lambda_, score), xytext=(0, 9), textcoords='offset points', ha='center', fontsize=8)
ax.set_xscale('log')
ax.set_xlabel('Pegasos regularization lambda')
ax.set_ylabel('Validation accuracy')
ax.set_title('Pegasos hyperparameter selection')
ax.set_ylim(-0.05, 1.08)
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


## 4. Compare the final models and learned word weights

Perceptron and Average Perceptron have no explicit regularization parameter, so only Pegasos is tuned with `lambda`. All three models learn a weight for each vocabulary feature. A positive weight pushes the score toward `+1`; a negative weight pushes it toward `-1`.


In [ ]:
final_models = {'Perceptron': perceptron(train_data, epochs=20), 'Average Perceptron': average_perceptron(train_data, epochs=20), 'Pegasos': pegasos(train_data, lambda_=best_lambda, epochs=40, seed=3)}
for name, weights in final_models.items():
    print(f'{name:20} train={accuracy(weights, train_data):.3f} validation={accuracy(weights, validation_data):.3f}')
word_weights = {name: {word: weights.get(index, 0.0) for word, index in vocabulary.items()} for name, weights in final_models.items()}
words = list(vocabulary)
x = np.arange(len(words))
width = 0.26
fig, ax = plt.subplots(figsize=(12, 5))
for offset, (name, weights) in zip((-width, 0, width), word_weights.items()):
    ax.bar(x + offset, [weights[w] for w in words], width, label=name)
ax.axhline(0, linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(words, rotation=45, ha='right')
ax.set_ylabel('Learned weight')
ax.set_title('Word weights learned by the three classifiers')
ax.legend()
plt.tight_layout()
plt.show()


## 5. Plot each classifier's decision boundary

Putting all three separators in one diagram can become visually crowded and makes the individual geometry harder to inspect. Instead, the same review points are shown in a separate figure for each classifier.

For this visualization, `excellent` is $x_1$ and `terrible` is $x_2$.

$$
\theta_1x_1+\theta_2x_2=0,
$$

so the boundary is

$$
x_2=-\frac{\theta_1}{\theta_2}x_1.
$$


In [ ]:
plot_reviews = [(+1, 'excellent product'), (+1, 'excellent value'), (+1, 'excellent quality'), (+1, 'excellent great'), (-1, 'terrible product'), (-1, 'terrible value'), (-1, 'terrible quality'), (-1, 'terrible poor')]
plot_vocabulary = {'excellent': 0, 'terrible': 1}
plot_X, plot_y = vectorize(plot_reviews, plot_vocabulary)
plot_data = list(zip(plot_y, plot_X))
plot_models = {'Perceptron': perceptron(plot_data, epochs=20), 'Average Perceptron': average_perceptron(plot_data, epochs=20), 'Pegasos': pegasos(plot_data, lambda_=best_lambda, epochs=40, seed=3)}
points = np.array([[features.get(0, 0.0), features.get(1, 0.0)] for features in plot_X])
x_values = np.linspace(-0.1, 1.1, 200)
for name, weights in plot_models.items():
    fig, ax = plt.subplots(figsize=(8, 7))
    for label, marker in [(1, 'o'), (-1, 'x')]:
        mask = plot_y == label
        ax.scatter(points[mask, 0], points[mask, 1], marker=marker, s=90, label='positive' if label == 1 else 'negative')
    theta1, theta2 = weights.get(0, 0.0), weights.get(1, 0.0)
    if abs(theta2) > 1e-12:
        ax.plot(x_values, -(theta1 / theta2) * x_values, linewidth=2, label='decision boundary')
    elif abs(theta1) > 1e-12:
        ax.axvline(0, linewidth=2, label='decision boundary')
    for (x1, x2), (_, review) in zip(points, plot_reviews):
        ax.annotate(review, (x1, x2), xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax.set_xlabel('excellent')
    ax.set_ylabel('terrible')
    ax.set_title(f'{name}: decision boundary on review feature space')
    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(-0.1, 1.1)
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()


## 6. Pegasos and the research connection

For a training example `(x, y)`, the hinge loss is

$$
\ell(\theta;(x,y))=\max\{0,1-y\theta^Tx\}.
$$

For a mini-batch $B_t$, the active examples are

$$
A_t=\{i\in B_t:y_i\theta_t^Tx_i<1\}.
$$

The implementation averages the active hinge-loss contribution over the full batch size and then applies the Pegasos projection. This connects the project implementation to the Pegasos algorithm studied in the course.


## 7. Experimental workflow and tests

For real data, keep the test set untouched while selecting the algorithm and hyperparameters. Build the vocabulary from training data only, then vectorize validation and test data with that vocabulary.

Run the reusable-classifier tests from `project_1`:

```bash
python3 -m unittest -v test_linear_classification.py
```

The module remains general-purpose: it accepts sparse labeled vectors and contains the three learning algorithms plus shared sparse-vector utilities. Review tokenization and vectorization are demonstrated here instead of being coupled to the module.
